In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-stage-3-2026")

print("Path to dataset files:", path)

In [ ]:
import torch
import torchvision.transforms as transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import kagglehub
import os

# 1. Download Dataset
path = kagglehub.dataset_download("mohammad2012191/q1-stage-3-2026")
print(f"Root dataset path: {path}")

   # trust me i was having some weird stuff happen i had to make sure
    #reflecting back i think i importent the dataset wrong or something and i just realized how much i did here, i should probably delete this but its working and don't have much time to fix it if it goes wrong again

    #note i deleted it andd try again and had the same error i have no idea why debugging such a problem

   # i made it autmatic find test and train
train_path = None
test_path = None

for root, dirs, files in os.walk(path):
    if "train" in dirs:
        train_path = os.path.join(root, "train")
    if "test" in dirs:
        test_path = os.path.join(root, "test")
    if train_path and test_path:
        break

# Verify paths were found, else fallback to root
if train_path is None:
    print("Warning: 'train' folder not found. Listing root contents:")
    print(os.listdir(path))
    train_path = os.path.join(path, "train")
if test_path is None:
    test_path = os.path.join(path, "test")

print(f"Resolved Train Path: {train_path}")
print(f"Resolved Test Path: {test_path}")

# Define Transformations
transform_train = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.RandomRotation(15),
    transforms.ToTensor()
])

transform_test = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.ToTensor()
])

try:
    train_dataset = ImageFolder(root=train_path, transform=transform_train)
    test_dataset = ImageFolder(root=test_path, transform=transform_test)

    train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

    print(f"Train samples: {len(train_dataset)}, Test samples: {len(test_dataset)}")
    print(f"Classes: {train_dataset.classes}")

    images, labels = next(iter(train_loader))
    classes = train_dataset.classes

    fig, axes = plt.subplots(1, 5, figsize=(15, 3))
    for i in range(5):
        ax = axes[i]
        img = images[i].permute(1, 2, 0)
        ax.imshow(img)
        ax.set_title(classes[labels[i]])
        ax.axis("off")
    plt.show()

except FileNotFoundError as e:
    print(f"Error loading dataset: {e}")
    print("Please check the 'Resolved Train Path' output above.")

In [ ]:
import torch.nn as nn

class PotatoCNN(nn.Module):
    def __init__(self, num_classes=3):
        super(PotatoCNN, self).__init__()

        # Convolutional Layers
        # Input: 3 x 32 x 32
        self.features = nn.Sequential(
            # Block 1
            nn.Conv2d(in_channels=3, out_channels=16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2), # Output: 16 x 16 x 16

            # Block 2
            nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2), # Output: 32 x 8 x 8

            # Block 3
            nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)  # Output: 64 x 4 x 4
        )
        #Note i just stuided all the math in theory just to get 3 or 4 questions in it, so i got to flex my understanding here

        # Fully Connected Layers
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 4 * 4, 128),
            nn.ReLU(),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = PotatoCNN(num_classes=len(train_dataset.classes)).to(device)
print(model)

In [ ]:
import torch.optim as optim
from tqdm import tqdm

# Define loss function and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=7, gamma=0.1) #
num_epochs = 15


def train_one_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    total_loss = 0
    correct = 0
    total = 0

    for images, labels in tqdm(dataloader, desc="Training"):
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()


        predictions = outputs.argmax(dim=1)
        correct += (predictions == labels).sum().item()
        total += labels.size(0)

    return total_loss / len(dataloader), 100 * correct / total

def validate(model, dataloader, criterion, device):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in dataloader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            total_loss += loss.item()

            predictions = outputs.argmax(dim=1)
            correct += (predictions == labels).sum().item()
            total += labels.size(0)

    return total_loss / len(dataloader), 100 * correct / total

res_train_losses = []
res_val_losses = []
res_train_accs = []
res_val_accs = []

for epoch in range(num_epochs):
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_acc = validate(model, test_loader, criterion, device)

    res_train_losses.append(train_loss)
    res_val_losses.append(val_loss)
    res_train_accs.append(train_acc)
    res_val_accs.append(val_acc)

    scheduler.step()

    print(f"Epoch {epoch+1}/{num_epochs} | LR: {scheduler.get_last_lr()[0]:.6f} | "
          f"Train Loss: {train_loss:.4f} Acc: {train_acc:.2f}% | "
          f"Val Loss: {val_loss:.4f} Acc: {val_acc:.2f}%")

plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(res_train_losses, label='Train Loss')
plt.plot(res_val_losses, label='Val Loss')
plt.title('CNN Loss')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(res_train_accs, label='Train Acc')
plt.plot(res_val_accs, label='Val Acc')
plt.title('CNN Accuracy')
plt.legend()
plt.show()

In [ ]:
from tqdm import tqdm

def train_one_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    total_loss = 0
    correct = 0
    total = 0

    for images, labels in tqdm(dataloader):
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    avg_loss = total_loss / len(dataloader)
    accuracy = 100 * correct / total
    return avg_loss, accuracy

def validate(model, dataloader, criterion, device):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in dataloader:
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)
            total_loss += loss.item()

            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    avg_loss = total_loss / len(dataloader)
    accuracy = 100 * correct / total
    return avg_loss, accuracy

In [ ]:
class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1):
        super(ResidualBlock, self).__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)

        self.shortcut = nn.Sequential()
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels)
            )

    def forward(self, x):
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out += self.shortcut(x)
        out = self.relu(out)
        return out

class PotatoResNet(nn.Module):
    def __init__(self, num_classes=3):
        super(PotatoResNet, self).__init__()
        self.in_channels = 16

        # Initial conv
        self.conv1 = nn.Conv2d(3, 16, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(16)
        self.relu = nn.ReLU(inplace=True)

        # Residual Layers
        self.layer1 = self._make_layer(16, 2, stride=1)
        self.layer2 = self._make_layer(32, 2, stride=2)
        self.layer3 = self._make_layer(64, 2, stride=2)

        # Classifier
        self.avg_pool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(64, num_classes)

    def _make_layer(self, out_channels, blocks, stride):
        layers = []
        layers.append(ResidualBlock(self.in_channels, out_channels, stride))
        self.in_channels = out_channels
        for _ in range(1, blocks):
            layers.append(ResidualBlock(out_channels, out_channels))
        return nn.Sequential(*layers)

    def forward(self, x):
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.layer1(out)
        out = self.layer2(out)
        out = self.layer3(out)
        out = self.avg_pool(out)
        out = out.view(out.size(0), -1)
        out = self.fc(out)
        return out

resnet_model = PotatoResNet(num_classes=len(train_dataset.classes)).to(device)
print(resnet_model)

optimizer_resnet = optim.Adam(resnet_model.parameters(), lr=0.001)
scheduler_resnet = optim.lr_scheduler.StepLR(optimizer_resnet, step_size=7, gamma=0.1)

res_train_losses = []
res_val_losses = []
res_train_accs = []
res_val_accs = []

print("Starting ResNet Training...")

for epoch in range(num_epochs):
    train_loss, train_acc = train_one_epoch(resnet_model, train_loader, criterion, optimizer_resnet, device)
    val_loss, val_acc = validate(resnet_model, test_loader, criterion, device)

    res_train_losses.append(train_loss)
    res_val_losses.append(val_loss)
    res_train_accs.append(train_acc)
    res_val_accs.append(val_acc)

    scheduler_resnet.step()

    print(f"Epoch {epoch+1}/{num_epochs} | LR: {scheduler_resnet.get_last_lr()[0]:.6f} | "
          f"Train Loss: {train_loss:.4f} Acc: {train_acc:.2f}% | "
          f"Val Loss: {val_loss:.4f} Acc: {val_acc:.2f}%")

# 5. Compare Results
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(res_train_losses, label='Train Loss')
plt.plot(res_val_losses, label='Val Loss')
plt.title('ResNet Loss')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(res_train_accs, label='Train Acc')
plt.plot(res_val_accs, label='Val Acc')
plt.title('ResNet Accuracy')
plt.legend()
plt.show()

In [ ]:
#BINGO